In [1]:
%pip install pandas numpy faker

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import random
from datetime import datetime

np.random.seed(42)
random.seed(42)

NUM_USERS = 50000
NUM_TRANSACTIONS = 800000
HIGH_RISK_USER_PERCENT = 0.08
BASE_FRAUD_RATE = 0.012   # base 1.2%

In [3]:
normal_countries = ["US", "UK", "Canada", "India", "Australia"]
high_risk_countries = ["Nigeria", "Russia", "Romania", "Indonesia", "Vietnam"]

all_countries = normal_countries + high_risk_countries

In [4]:
users = pd.DataFrame({
    "user_id": np.arange(1, NUM_USERS + 1),
    "signup_date": pd.to_datetime(
        np.random.choice(
            pd.date_range("2021-01-01", "2024-01-01"),
            NUM_USERS
        )
    ),
    "country": np.random.choice(
        normal_countries + high_risk_countries,
        NUM_USERS,
        p=[0.18,0.15,0.14,0.15,0.10, 0.08,0.07,0.06,0.04,0.03]
    ),
    "prior_chargebacks": np.random.poisson(0.25, NUM_USERS)
})

users["account_age_days"] = (
    pd.to_datetime("2025-01-01") - users["signup_date"]
).dt.days

users["high_risk_user"] = np.random.choice(
    [0,1],
    NUM_USERS,
    p=[1-HIGH_RISK_USER_PERCENT, HIGH_RISK_USER_PERCENT]
)

users.head()

,user_id,signup_date,country,prior_chargebacks,account_age_days,high_risk_user
0,1,2023-05-11,Romania,0,601,0
1,2,2024-01-01,Nigeria,1,366,0
2,3,2023-11-11,India,0,417,0
3,4,2021-05-02,US,1,1340,0
4,5,2022-04-12,UK,0,995,0


In [5]:
payment_methods = ["Card", "Wallet", "Bank Transfer", "Crypto"]
merchant_categories = [
    "Electronics",
    "Gaming",
    "Digital Services",
    "Fashion",
    "Luxury Goods",
    "Travel",
    "Grocery",
    "Subscriptions"
]
device_types = ["Mobile", "Desktop", "Tablet"]
currencies = ["USD", "INR", "GBP", "EUR", "AUD"]

transactions = pd.DataFrame({
    "transaction_id": np.arange(1, NUM_TRANSACTIONS + 1),
    "user_id": np.random.choice(users["user_id"], NUM_TRANSACTIONS),
    "transaction_time": pd.to_datetime(
        np.random.choice(
            pd.date_range("2024-01-01", "2025-01-01", freq="min"),
            NUM_TRANSACTIONS
        )
    ),
    "amount": np.round(np.maximum(np.random.gamma(2.2, 85, NUM_TRANSACTIONS), 1), 2),
    "currency": np.random.choice(currencies, NUM_TRANSACTIONS, p=[0.4,0.25,0.15,0.1,0.1]),
    "payment_method": np.random.choice(payment_methods, NUM_TRANSACTIONS, p=[0.6,0.18,0.14,0.08]),
    "merchant_category": np.random.choice(merchant_categories, NUM_TRANSACTIONS),
    "device_type": np.random.choice(device_types, NUM_TRANSACTIONS, p=[0.65,0.30,0.05]),
    "ip_country": np.random.choice(all_countries, NUM_TRANSACTIONS),
    "status": np.random.choice(["Success", "Failed"], NUM_TRANSACTIONS, p=[0.92,0.08])
})

In [6]:
transactions = transactions.merge(
    users[["user_id", "country", "high_risk_user"]],
    on="user_id",
    how="left"
)

transactions["cross_border"] = (
    transactions["ip_country"] != transactions["country"]
).astype(int)

In [7]:
risk_score = (
    (transactions["high_risk_user"] * 0.30) +
    (transactions["cross_border"] * 0.20) +
    (transactions["payment_method"] == "Card").astype(int) * 0.15 +
    (transactions["payment_method"] == "Crypto").astype(int) * 0.25 +
    (transactions["merchant_category"] == "Luxury Goods").astype(int) * 0.10 +
    (transactions["amount"] > 600).astype(int) * 0.20
)

transactions["fraud_probability"] = BASE_FRAUD_RATE + risk_score * 0.05

transactions["is_chargeback"] = (
    np.random.rand(NUM_TRANSACTIONS) < transactions["fraud_probability"]
).astype(int)

In [8]:
# ==========================
# CHARGEBACK TABLE WITH LATENCY
# ==========================

chargebacks = transactions[transactions["is_chargeback"] == 1].copy()

# Add latency (7–45 days)
latency_days = np.random.randint(7, 46, len(chargebacks))

chargebacks["chargeback_date"] = (
    chargebacks["transaction_time"] +
    pd.to_timedelta(latency_days, unit="D")
)

chargebacks = chargebacks[[
    "transaction_id",
    "user_id",
    "chargeback_date",
    "amount",
    "payment_method",
    "merchant_category"
]]

chargebacks["reason"] = np.random.choice(
    [
        "Fraud - Unauthorized",
        "Fraud - Card Not Present",
        "Duplicate Charge",
        "Product Not as Described",
        "Service Not Rendered",
        "Subscription Cancellation"
    ],
    len(chargebacks),
    p=[0.55,0.13,0.09,0.08,0.07,0.08]
)

chargebacks.reset_index(drop=True, inplace=True)

In [9]:
transactions.drop(columns=[
    "high_risk_user",
    "country",
    "cross_border",
    "fraud_probability"
], inplace=True)

In [10]:
# ==========================
# ELITE NULL INJECTION
# ==========================

def inject_nulls(df, column, percentage):
    n = int(len(df) * percentage)
    null_indices = np.random.choice(df.index, n, replace=False)
    df.loc[null_indices, column] = None

# Inject controlled NULLs
inject_nulls(transactions, "ip_country", 0.02)
inject_nulls(transactions, "device_type", 0.01)
inject_nulls(transactions, "merchant_category", 0.015)

print("NULL Injection Complete")

NULL Injection Complete


In [11]:
print("Users:", len(users))
print("Transactions:", len(transactions))
print("Chargebacks:", len(chargebacks))
print("Fraud Rate:", round(len(chargebacks)/len(transactions)*100, 2), "%")

print("\nNULL CHECK:")
print(users.isnull().sum())
print(transactions.isnull().sum())
print(chargebacks.isnull().sum())

Users: 50000
Transactions: 800000
Chargebacks: 22578
Fraud Rate: 2.82 %

NULL CHECK:
user_id              0
signup_date          0
country              0
prior_chargebacks    0
account_age_days     0
high_risk_user       0
dtype: int64
transaction_id           0
user_id                  0
transaction_time         0
amount                   0
currency                 0
payment_method           0
merchant_category    12000
device_type           8000
ip_country           16000
status                   0
is_chargeback            0
dtype: int64
transaction_id       0
user_id              0
chargeback_date      0
amount               0
payment_method       0
merchant_category    0
reason               0
dtype: int64


In [12]:
# ==========================
# CLEAN ATO BEHAVIOR SHIFT
# ==========================

ato_users = np.random.choice(
    users[users["prior_chargebacks"] == 0]["user_id"],
    size=int(NUM_USERS * 0.02),
    replace=False
)

transactions["is_ato_user"] = transactions["user_id"].isin(ato_users).astype(int)

ato_date = pd.to_datetime("2024-06-01")

mask_ato_after = (
    (transactions["is_ato_user"] == 1) &
    (transactions["transaction_time"] >= ato_date)
)

# Increase fraud probability instead of overwrite
additional_risk = np.random.rand(mask_ato_after.sum()) < 0.07

transactions.loc[mask_ato_after, "is_chargeback"] = np.maximum(
    transactions.loc[mask_ato_after, "is_chargeback"],
    additional_risk.astype(int)
)

In [13]:
# ==========================
# VELOCITY BURSTS
# ==========================

burst_users = np.random.choice(users["user_id"], 800, replace=False)

for user in burst_users:
    user_tx = transactions[transactions["user_id"] == user]
    
    if len(user_tx) > 10:
        burst_indices = user_tx.sample(6).index
        
        base_time = pd.to_datetime("2024-11-15 12:00:00")
        
        for i, idx in enumerate(burst_indices):
            transactions.loc[idx, "transaction_time"] = base_time + pd.Timedelta(seconds=i*20)
            transactions.loc[idx, "amount"] *= (1 + i*0.2)

print("Velocity bursts injected")

Velocity bursts injected


In [14]:
# ==========================
# FRAUD CAMPAIGN SPIKE
# ==========================

campaign_mask = (
    (transactions["transaction_time"].dt.month == 9) &
    (transactions["transaction_time"].dt.year == 2024) &
    (transactions["merchant_category"] == "Luxury Goods")
)

transactions.loc[campaign_mask, "is_chargeback"] = (
    np.random.rand(campaign_mask.sum()) < 0.15
).astype(int)

print("Fraud campaign spike applied")

Fraud campaign spike applied


In [15]:
print("Final Fraud Rate:",
      round(transactions["is_chargeback"].mean()*100,2), "%")

Final Fraud Rate: 3.01 %


In [16]:
users_df = users.copy()
transactions_df = transactions.copy()
chargebacks_df = chargebacks.copy()

In [17]:
users_df = users_df.sort_values("user_id").reset_index(drop=True)

transactions_df = transactions_df.sort_values(
    ["transaction_time", "transaction_id"]
).reset_index(drop=True)

chargebacks_df = chargebacks_df.sort_values(
    ["chargeback_date", "transaction_id"]
).reset_index(drop=True)

In [18]:
print("Users:", len(users_df))
print("Transactions:", len(transactions_df))
print("Chargebacks:", len(chargebacks_df))

Users: 50000
Transactions: 800000
Chargebacks: 22578


In [19]:
transactions["amount"] = transactions["amount"].round(2)
chargebacks["amount"] = chargebacks["amount"].round(2)

In [21]:
users.to_csv(
    "users.csv",
    index=False,
    encoding="utf-8",
    lineterminator="\n"
)

transactions.to_csv(
    "transactions.csv",
    index=False,
    encoding="utf-8",
    lineterminator="\n",
    float_format="%.2f"
)

chargebacks.to_csv(
    "chargebacks.csv",
    index=False,
    encoding="utf-8",
    lineterminator="\n",
    float_format="%.2f"
)